PySpark Scenario-Based Interview Series:
Day 1  Data Cleaning & Filtering (Most Common Interview Scenario)

##### InterviewScenario
You are working as a PySpark Data Engineer.
You receive daily customer transaction data from multiple sources.

The data has:
Null values
Duplicate records
Invalid amounts
Wrong data types
You are asked to clean the data and prepare it for analytics.

Sample Input Data:

##InterviewApproach :
Understand business rules
Remove duplicates
Handle null values
Filter invalid records
Ensure correct data types
Prepare clean dataset for downstream systems

I ingest the raw data into Spark, remove duplicates using a business key, handle null values to avoid downstream failures, separate invalid records for auditing, and finally write clean data in Parquet format for analytics or OLAP consumption.


customer_id | transaction_id | amount | country | transaction_date
---------------------------------------------------------------
 101      | T001       | 500  | US    | 2024-01-01
 102      | T002       | null  | IN    | 2024-01-01
 103      | T003       | -100 | US    | 2024-01-02
 101      | T001       | 500  | US    | 2024-01-01
 104      | T004       | 800  | null   | 2024-01-03


In [25]:
from sqlite3.dbapi2 import Date

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

In [2]:
spark = SparkSession.builder \
    .appName("MySQL JDBC Example") \
    .config("spark.jars", "/Users/nikkinani/Downloads/mysql-connector-j-8.2.0/mysql-connector-j-8.2.0.jar") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/01/09 14:00:08 WARN Utils: Your hostname, Nikkis-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.17.65.4 instead (on interface en0)
26/01/09 14:00:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/01/09 14:00:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/09 14:00:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
data = [
    (101, "T001", 500, "US", "2024-01-01"),
    (102, "T002", None, "IN", "2024-01-01"),
    (103, "T003", -100, "US", "2024-01-02"),
    (101, "T001", 500, "US", "2024-01-01"),
    (104, "T004", 800, None, "2024-01-03")
]

columns = ["customer_id", "transaction_id", "amount", "country", "transaction_date"]

In [4]:
raw_df = spark.createDataFrame(data, columns)
raw_df.show()

+-----------+--------------+------+-------+----------------+
|customer_id|transaction_id|amount|country|transaction_date|
+-----------+--------------+------+-------+----------------+
|        101|          T001|   500|     US|      2024-01-01|
|        102|          T002|  NULL|     IN|      2024-01-01|
|        103|          T003|  -100|     US|      2024-01-02|
|        101|          T001|   500|     US|      2024-01-01|
|        104|          T004|   800|   NULL|      2024-01-03|
+-----------+--------------+------+-------+----------------+



In [33]:
# drop duplicates
dedup_df = raw_df.dropDuplicates(["customer_id"])

`avg_amount = raw_df.agg(F.avg("amount").alias("avg_amount")).first()["avg_amount"]


# handle null values
filled_df = dedup_df.fillna({
    "amount" : avg_amount,
    "amount" : F.abs(F.col("amount"),
    "country" : "UNKNOWN"})
#typecast the date col from string to date

typecasted_df = filled_df.withColumn("transaction_date", F.to_date("transaction_date", "yyyy-MM-dd"))
typecasted_df.show()

+-----------+--------------+------+-------+----------------+
|customer_id|transaction_id|amount|country|transaction_date|
+-----------+--------------+------+-------+----------------+
|        101|          T001|   500|     US|      2024-01-01|
|        102|          T002|   425|     IN|      2024-01-01|
|        103|          T003|  -100|     US|      2024-01-02|
|        104|          T004|   800|UNKNOWN|      2024-01-03|
+-----------+--------------+------+-------+----------------+



In [34]:
typecasted_df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- country: string (nullable = false)
 |-- transaction_date: date (nullable = true)

